# Using ATLAS Open Data (event generation) for phenomenology studies

This notebook walks through the workflow for using **ATLAS Open Data event-generation samples**
(generator-level HepMC events) as simulated backgrounds/references for your own phenomenology study.
It is meant to run inside a **SWAN** session at CERN (or on the ESCAPE VRE) with a recent
**LCG software environment** loaded, which provides `Rivet`, `YODA`, `HepMC`, `Sherpa`, ...

**What we do here**

1. Discover the relevant open-data samples and their metadata with the [`atlasopenmagic`](https://github.com/atlas-outreach-data-tools/atlasopenmagic) package.
2. Obtain a HepMC event file (either directly from EOS on SWAN, or by downloading it).
3. Run a standard **Rivet** analysis (`MC_ZINC`, Drell-Yan / inclusive Z) on the events.
4. Plot the results with `rivet-mkhtml`, and also **inline in the notebook** with `YODA` + `matplotlib`.
5. Build and run a **custom Rivet analysis** plugin.
6. Download the resulting histograms for use in your own study.

> The concrete example is dataset **601189** — `PhPy8EG_AZNLO_Zee`, a POWHEG+Pythia8 Drell-Yan
> $Z\to ee$ sample at $\sqrt{s}=13.6$ TeV — but every step is parameterised so you can swap in any
> other Dataset ID (DSID).

## 0. Environment

On SWAN, pick a recent **LCG release** (e.g. one of the `LCG_10x` / `dev` stacks) so that Rivet, YODA
and HepMC are on the path. The cell below checks the tool versions. If any command is missing, go back to
the SWAN configuration and choose a software stack that bundles the MC generator/analysis tools.

In [ ]:
# --- Environment sanity check (run in a SWAN terminal or here with the ! shell escape) ---
!echo "Rivet:"; rivet --version 2>/dev/null || echo "  rivet not found - load an LCG stack that provides it"
!echo "YODA:";  python3 -c "import yoda; print(' ', yoda.version())" 2>/dev/null || yodals --version 2>/dev/null || echo "  yoda not found"
!echo "HepMC:"; HepMC3-config --version 2>/dev/null || echo "  (HepMC lib present via LCG)"

## 1. Discover datasets with `atlasopenmagic`

`atlasopenmagic` is the official helper that exposes the ATLAS Open Data metadata catalogue: releases,
per-dataset physics parameters (cross-section, generator, keywords, ...) and the file URLs.

**On CERN SWAN / the ESCAPE VRE you do not need to `pip install` it** — it ships in CVMFS. The helper
`setup_notebook.py` at `/cvmfs/sw.escape.eu/atlasopenmagic/latest` puts the package on `sys.path` and its
CLI tools on `PATH`. It requires a **Python ≥ 3.10** kernel, so choose a recent LCG / `dev` stack in the
SWAN start form. The cell below uses CVMFS first and only falls back to `pip` if CVMFS isn't mounted.

In [ ]:
# Preferred on CERN SWAN / ESCAPE VRE: load atlasopenmagic from the CVMFS distribution (no pip needed).
# Requires a Python >= 3.10 kernel (pick a recent LCG / dev stack in the SWAN configuration form).
AOM_CVMFS = "/cvmfs/sw.escape.eu/atlasopenmagic/latest"

import os, sys
try:
    import atlasopenmagic  # already importable? (e.g. provided by the LCG view)
except ImportError:
    if os.path.isdir(AOM_CVMFS):
        sys.path.insert(0, AOM_CVMFS)                 # so we can import the setup helper
        from setup_notebook import setup_atlasopenmagic
        setup_atlasopenmagic(AOM_CVMFS)               # adds <home>/lib/pythonX.Y/site-packages and <home>/bin
    else:
        !{sys.executable} -m pip install --user atlasopenmagic   # fallback if CVMFS is not available

import atlasopenmagic as atom
print("Using atlasopenmagic from:", atom.__file__)

In [ ]:
import atlasopenmagic as atom

# See what open-data releases exist
atom.available_releases()

We use the **event-generation** research release at 13.6 TeV. (There is also a 13 TeV twin,
`2025r-evgen-13tev`.) These releases contain generator-level **HepMC** events — exactly what Rivet
consumes — rather than reconstructed physics objects.

In [ ]:
RELEASE = "2025r-evgen-13p6tev"   # or "2025r-evgen-13tev"
atom.set_release(RELEASE)
print("Active release:", atom.get_current_release())

### Browse by physics keyword

Each sample is tagged with keywords (`z`, `top`, `higgs`, `diboson`, ...). You can list them and then
find matching datasets. Here we look for the Drell-Yan $Z\to ee$ samples.

In [ ]:
# All keywords available in this release
print(atom.available_keywords())

# Find Z->ee samples by their short physics name
atom.match_metadata("physics_short", "Zee")

### Inspect the metadata of our chosen sample

We pick **601189** (`PhPy8EG_AZNLO_Zee`). The metadata gives us everything we need to normalise the
sample later: cross-section, filter efficiency, k-factor and the number of generated events.

In [ ]:
DSID = "601189"   # PhPy8EG_AZNLO_Zee  (POWHEG+Pythia8 Drell-Yan Z->ee, sqrt(s)=13.6 TeV)

meta = atom.get_metadata(DSID)
for k in ["physics_short", "description", "generator", "GenTune",
          "CoMEnergy", "cross_section_pb", "kFactor", "genFiltEff",
          "nEvents", "hepmc_version", "keywords"]:
    print(f"{k:20s}: {meta.get(k)}")

# Effective cross-section for normalisation (pb):  sigma * k-factor * filter-eff
xsec_pb = float(meta["cross_section_pb"]) * float(meta["kFactor"]) * float(meta["genFiltEff"])
print("\nEffective cross-section [pb]:", xsec_pb)

## 2. Get the file URLs

`get_urls` returns the list of HepMC files for the dataset. Three protocols are available:

- `https` — plain web download (works anywhere),
- `root`  — XRootD streaming,
- `eos`   — a POSIX `/eos/...` path that you can read **directly on SWAN** without downloading.

On SWAN the `eos` path is the fast option: the file is already on the mounted EOS space.

In [ ]:
urls_https = atom.get_urls(DSID, protocol="https")
urls_eos   = atom.get_urls(DSID, protocol="eos")

print(f"{len(urls_https)} files in dataset {DSID}\n")
print("First HTTPS URL:\n ", urls_https[0])
print("\nFirst EOS path (use this directly on SWAN):\n ", urls_eos[0])

## 3. Obtain one HepMC event file

The open-data event files are shipped as gzipped **tar archives** (`HEPMC.*.tar.gz`) containing an ASCII
HepMC record. Below are two ways to get a local `.hepmc` file to analyse.

**Path A — on SWAN (recommended):** the files already live on EOS, so you only need to un-tar one of them.
**Path B — anywhere:** download a single file over HTTPS, then un-tar it.

Both paths end with the variable `HEPMC_FILE` pointing at an ASCII HepMC file that Rivet can read.
We only take **one** file here so the example runs quickly — add more for higher statistics.

In [ ]:
import os, tarfile, urllib.request, shutil

workdir = os.path.abspath("evgen_work")
os.makedirs(workdir, exist_ok=True)

def plain_url(u):
    # atlasopenmagic returns fsspec chained URLs, e.g. "simplecache::https://...".
    # Keep only the real transport URL (the part after the last "::").
    return u.split("::")[-1]

def to_local(url, dest_dir):
    # Return a local path to `url`, downloading over HTTPS if it isn't already present.
    u = plain_url(url)
    local = os.path.join(dest_dir, os.path.basename(u))
    if not os.path.exists(local):
        print("Downloading", u, "...")
        urllib.request.urlretrieve(u, local)
    return local

def untar_hepmc(tar_path, dest):
    # Extract an ATLAS open-data HEPMC tarball; return the path to the ASCII HepMC file.
    with tarfile.open(tar_path) as tf:
        members = [m for m in tf.getmembers() if m.isfile()]
        tf.extractall(dest, members=members)
        extracted = [os.path.join(dest, m.name) for m in members]
    # the HepMC ASCII record is the (single) largest extracted file
    extracted.sort(key=os.path.getsize, reverse=True)
    return extracted[0]

# ---- Path A: SWAN / EOS (uncomment on SWAN) --------------------------------
# The eos protocol gives a POSIX /eos/... path already on the SWAN-mounted storage,
# so just copy it locally (no download):
# src_tar   = plain_url(urls_eos[0])
# local_tar = os.path.join(workdir, os.path.basename(src_tar))
# shutil.copy(src_tar, local_tar)

# ---- Path B: HTTPS download (works off SWAN) -------------------------------
local_tar = to_local(urls_https[0], workdir)

HEPMC_FILE = untar_hepmc(local_tar, workdir)
print("HepMC file ready:", HEPMC_FILE, f"({os.path.getsize(HEPMC_FILE)/1e6:.1f} MB)")

## 4. Run a standard Rivet analysis (`MC_ZINC`)

`MC_ZINC` is Rivet's built-in inclusive-Z (Drell-Yan) analysis — a natural match for our $Z\to ee$ sample.
It produces distributions such as the Z transverse momentum, rapidity and mass. Rivet reads the
cross-section directly from the HepMC `GenCrossSection` record; we also pass it explicitly for safety.

In [ ]:
YODA_OUT = os.path.join(workdir, "Zee_MC_ZINC.yoda.gz")

# --cross-section is in pb; -a selects the analysis; -o is the output histogram file
!rivet "{HEPMC_FILE}" -a MC_ZINC --cross-section={xsec_pb} -o "{YODA_OUT}"
print("\nWrote:", YODA_OUT)

## 5. Make plots with `rivet-mkhtml`

`rivet-mkhtml` turns one (or several) YODA files into a browsable set of plots. It writes a directory
`rivet-plots/` containing an `index.html`.

In [ ]:
PLOTDIR = os.path.join(workdir, "rivet-plots")

# rivet-mkhtml renders labels with LaTeX. Some SWAN TeX Live setups choke on it
# (e.g. "latex was not able to process ... 'lp'"). Best-effort: tell matplotlib not
# to use LaTeX for the subprocess plot scripts. If it still fails, use the inline
# plot in the next cell (identical physics, no LaTeX) - the .yoda.gz is already written.
import os
_rc = os.path.join(workdir, "matplotlibrc")
open(_rc, "w").write("text.usetex: False\n")
os.environ["MATPLOTLIBRC"] = _rc

!rivet-mkhtml "{YODA_OUT}" -o "{PLOTDIR}" || echo "rivet-mkhtml hit an error (likely LaTeX) - see the inline plot below."
print("If it succeeded, open in the SWAN file browser:", os.path.join(PLOTDIR, "index.html"))

On SWAN you can open `rivet-plots/index.html` straight from the Jupyter file browser (double-click).
Because the LaTeX step above can be fragile on SWAN, the cell below is the **reliable way to view results
inline**: it reads the YODA output with the `yoda` Python bindings and draws it with `matplotlib`
(no LaTeX). It is written to work with both YODA 2.x (your SWAN stack) and older YODA 1.x.

In [ ]:
import yoda
import numpy as np
import matplotlib.pyplot as plt   # SWAN shows figures inline by default

aos = yoda.read(YODA_OUT)

def pick(suffix):
    # Return the (nominal) Histo1D whose path ends with `suffix`.
    for k, v in aos.items():
        if k.endswith(suffix) and isinstance(v, yoda.Histo1D):
            return v
    return None

def hist_xy(h):
    # Bin centres, heights (dsigma/dx) and errors - robust across YODA versions.
    # YODA 2.x: vectorised accessors on the histogram itself
    try:
        y = np.asarray(h.heights(), dtype=float)
        x = np.asarray(h.xMids() if hasattr(h, "xMids")
                       else [b.xMid() for b in h.bins()], dtype=float)
        e = np.asarray(h.heightErrs(), dtype=float) if hasattr(h, "heightErrs") \
            else np.zeros_like(y)
        return x, y, e
    except Exception:
        pass
    # YODA 1.x: per-bin accessors
    try:
        x = np.array([b.xMid()      for b in h.bins()], dtype=float)
        y = np.array([b.height()    for b in h.bins()], dtype=float)
        e = np.array([b.heightErr() for b in h.bins()], dtype=float)
        return x, y, e
    except Exception:
        pass
    # Last resort: convert to a Scatter and read the points
    s = h.mkScatter()
    pts = s.points()
    x = np.array([p.x() for p in pts]); y = np.array([p.y() for p in pts])
    try:
        e = np.array([max(abs(v) for v in p.yErrs()) for p in pts])
    except Exception:
        e = np.zeros_like(y)
    return x, y, e

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (suffix, xlabel, logy) in zip(
        axes, [("/Z_mass", r"$m_{ee}$ [GeV]", False),
               ("/Z_pT",   r"$p_{T}^{Z}$ [GeV]", True)]):
    h = pick(suffix)
    if h is None:
        ax.set_title(f"(no {suffix} in output)"); continue
    x, y, ye = hist_xy(h)
    ax.errorbar(x, y, yerr=ye, fmt="o", ms=3, lw=1)
    if logy: ax.set_yscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"d$\sigma$/d$x$ [pb/GeV]")
    ax.set_title(f"MC_ZINC {suffix.strip('/')}  (DSID {DSID})")
fig.tight_layout()
plt.show()

## 6. Run a *custom* Rivet analysis

If the built-in analyses are not enough, write your own routine. The workflow is: scaffold a template,
edit the C++ `analyze()` method, build the plugin, then point Rivet at it.

On your **local computer** you would copy your files up to SWAN, e.g.:

```bash
scp My_Analysis.cc My_Analysis.info My_Analysis.plot <user>@swan:...
```

Inside the SWAN session you build and run the plugin:

In [ ]:
# Scaffold a template routine (creates My_Analysis.{cc,info,plot}) - skip if you already uploaded yours
ANADIR = os.path.join(workdir, "my_analysis")
os.makedirs(ANADIR, exist_ok=True)
os.chdir(ANADIR)
if not os.path.exists("My_Analysis.cc"):
    !rivet-mkanalysis My_Analysis

# Build the plugin -> produces RivetMy_Analysis.so
!rivet-buildplugin My_Analysis.cc

# Run it: RIVET_ANALYSIS_PATH must include the dir with the built .so
env = f'RIVET_ANALYSIS_PATH="{ANADIR}:$RIVET_ANALYSIS_PATH"'
!{env} rivet "{HEPMC_FILE}" -a My_Analysis --cross-section={xsec_pb} -o "{os.path.join(workdir,'My_Analysis.yoda.gz')}"
os.chdir(workdir)

## 7. Download / export your results

Your histograms live in the `*.yoda.gz` files. A few ways to take them with you:

- **From the Jupyter file browser:** right-click the `.yoda.gz` (or the `rivet-plots` folder) → *Download*.
- **Convert to ROOT** for use in a ROOT-based analysis: `yoda2root`.
- **Read them back in Python** (as in step 5) and save `numpy`/`csv`/`matplotlib` outputs.

You can then use these as a reference or background template in your own phenomenology study.

In [ ]:
# List the artefacts produced, and optionally convert the main result to a ROOT file
!ls -lh "{workdir}"/*.yoda.gz
!yoda2root "{YODA_OUT}" 2>/dev/null && echo "Wrote ROOT file next to the YODA output" || echo "yoda2root not available - use the YODA file directly"

---
### References

- ATLAS Open Data — event generation release (record 160000): https://opendata.cern.ch/record/160000
- `atlasopenmagic` package & docs: https://github.com/atlas-outreach-data-tools/atlasopenmagic · https://opendata.atlas.cern/docs/data/atlasopenmagic
- Rivet: https://rivet.hepforge.org · YODA: https://yoda.hepforge.org · HepMC: https://hepmc.web.cern.ch

*Example dataset:* DSID **601189** `PhPy8EG_AZNLO_Zee`, POWHEG+Pythia8 Drell-Yan $Z\to ee$,
$\sqrt{s}=13.6$ TeV, $\sigma \approx 1998.8$ pb (HepMC2). Swap `DSID`/`RELEASE` above to use any other sample.